In [53]:
import pyspark.sql.functions as F

from pyspark.sql import Window, SparkSession
from pyspark.sql.functions import col, count, sum, avg, desc, round, percent_rank, lit


In [2]:
spark = SparkSession.builder.appName("DataFrameGroupingAggregation").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/19 19:04:30 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/19 19:04:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 19:04:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [
    (1, "Иван Петров", "2023-09-15", "Ноутбук", "Электроника", 1, 999.99, "Курьер"),
    (2, "Анна Сидорова", "2023-09-16", "Смартфон", "Электроника", 2, 599.50, "Самовывоз"),
    (3, "Елена Иванова", "2023-09-17", "Наушники", "Аксессуары", 3, 79.99, "Почта"),
    (4, "Анна Козлова", "2023-09-18", "Смартфон", "Электроника", 1, 699.99, "Курьер"),
    (5, "Петр Смирнов", "2023-09-19", "Планшет", "Электроника", 1, 449.99, "Самовывоз"),
    (6, "Иван Петров", "2023-09-20", "Чехол", "Аксессуары", 2, 19.99, "Почта"),
    (7, "Анна Сидорова", "2023-09-20", "Зарядное устройство", "Аксессуары", 1, 29.99, "Самовывоз"),
    (8, "Елена Иванова", "2023-09-21", "Смартфон", "Электроника", 1, 799.99, "Курьер")
]

In [4]:
schema = ["order_id", "customer_name", "order_date", "product_name", "category", "quantity", "price", "delivery_method"]
df = spark.createDataFrame(data, schema)

In [5]:
df.show()

+--------+-------------+----------+-------------------+-----------+--------+------+---------------+
|order_id|customer_name|order_date|       product_name|   category|quantity| price|delivery_method|
+--------+-------------+----------+-------------------+-----------+--------+------+---------------+
|       1|  Иван Петров|2023-09-15|            Ноутбук|Электроника|       1|999.99|         Курьер|
|       2|Анна Сидорова|2023-09-16|           Смартфон|Электроника|       2| 599.5|      Самовывоз|
|       3|Елена Иванова|2023-09-17|           Наушники| Аксессуары|       3| 79.99|          Почта|
|       4| Анна Козлова|2023-09-18|           Смартфон|Электроника|       1|699.99|         Курьер|
|       5| Петр Смирнов|2023-09-19|            Планшет|Электроника|       1|449.99|      Самовывоз|
|       6|  Иван Петров|2023-09-20|              Чехол| Аксессуары|       2| 19.99|          Почта|
|       7|Анна Сидорова|2023-09-20|Зарядное устройство| Аксессуары|       1| 29.99|      Самовывоз|


In [10]:
df_orders_per_customer = (
    df
        .groupBy("customer_name")
        .count()
        .withColumnRenamed("count", "total_orders")
)
df_orders_per_customer.show()

+-------------+------------+
|customer_name|total_orders|
+-------------+------------+
|Анна Сидорова|           2|
|  Иван Петров|           2|
|Елена Иванова|           2|
| Анна Козлова|           1|
| Петр Смирнов|           1|
+-------------+------------+



In [21]:
df_total_by_category = (
    df
        .groupBy("category")
        .agg(F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("total_value"))
)
df_total_by_category.show()

+-----------+-----------+
|   category|total_value|
+-----------+-----------+
|Электроника|    4148.96|
| Аксессуары|     309.94|
+-----------+-----------+



In [25]:
df_avg_check_per_day = (
    df
        .groupBy("order_date")
        .agg(F.round(F.avg(F.col("price") * F.col("quantity")), 2).alias("average_check"))
)
df_avg_check_per_day.show()

+----------+-------------+
|order_date|average_check|
+----------+-------------+
|2023-09-16|       1199.0|
|2023-09-15|       999.99|
|2023-09-18|       699.99|
|2023-09-17|       239.97|
|2023-09-20|        34.99|
|2023-09-19|       449.99|
|2023-09-21|       799.99|
+----------+-------------+



In [32]:
df_top_products = (
    df
        .groupBy("product_name")
        .agg(F.sum(F.col("quantity")).alias("total_quantity"))
        .orderBy(F.col("total_quantity").desc())
        .limit(3)
)
df_top_products.show()

+------------+--------------+
|product_name|total_quantity|
+------------+--------------+
|    Смартфон|             4|
|    Наушники|             3|
|       Чехол|             2|
+------------+--------------+



In [58]:
window_spec = Window.partitionBy()

df_delivery_percentage = (
    df
        .groupBy("delivery_method")
        .agg(F.count("order_id").alias("order_count"))
        .withColumn("percentage", F.round(F.col("order_count") / F.sum("order_count").over(window_spec) * 100, 2))
        .orderBy("delivery_method")

)

df_delivery_percentage.show()

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

In [57]:
spark.stop()